# Fast Fourier Transform — Worked Example

Companion notebook to the note **"Fast Fourier Transform Explained from the DFT"**: 
<https://jienweng.github.io/notes/fft-sound-wave/>

Everything here is from scratch with NumPy, plus a check against `numpy.fft`. 
Run top to bottom.

*Adapted material licensed for reuse; this notebook is by Jien Weng.*

## 1. Naive DFT and recursive Cooley–Tukey FFT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def naive_dft(x):
    x = np.asarray(x, dtype=complex)
    N = x.shape[0]
    n = np.arange(N)
    k = n.reshape((N, 1))
    M = np.exp(-2j * np.pi * k * n / N)
    return M @ x

def fft_recursive(x):
    x = np.asarray(x, dtype=complex)
    N = x.shape[0]
    if N & (N - 1):
        raise ValueError('Length of input must be a power of 2.')
    if N <= 2:
        return naive_dft(x)
    X_even = fft_recursive(x[0::2])
    X_odd = fft_recursive(x[1::2])
    k = np.arange(N // 2)
    twiddle = np.exp(-2j * np.pi * k / N)
    return np.concatenate([X_even + twiddle * X_odd,
                           X_even - twiddle * X_odd])

## 2. Sanity check: N = 4 and agreement with NumPy

In [ ]:
x = np.array([1, 2, 3, 4], dtype=complex)
print('numpy  :', np.fft.fft(x))
print('naive  :', naive_dft(x))
print('fft_rec:', fft_recursive(x))

rng = np.random.default_rng(0)
y = rng.standard_normal(1024)
print(np.allclose(naive_dft(y), np.fft.fft(y)))
print(np.allclose(fft_recursive(y), np.fft.fft(y)))

## 3. A real sound wave: three tones in, three peaks out

In [ ]:
sample_rate = 8000
duration = 1.0
N = int(sample_rate * duration)
t = np.arange(N) / sample_rate

x = (1.0 * np.sin(2 * np.pi * 220 * t)
     + 0.6 * np.sin(2 * np.pi * 440 * t)
     + 0.3 * np.sin(2 * np.pi * 880 * t))
x += 0.05 * np.random.default_rng(0).standard_normal(N)

plt.figure(figsize=(7, 3))
plt.plot(t[:400] * 1000, x[:400])
plt.xlabel('Time (ms)'); plt.ylabel('Amplitude')
plt.title('Time domain'); plt.grid(True, alpha=0.3); plt.show()

In [ ]:
def next_pow2(n):
    return 1 << (n - 1).bit_length()

N_pad = next_pow2(N)
x_pad = np.zeros(N_pad); x_pad[:N] = x
X = fft_recursive(x_pad)
magnitude = np.abs(X) / N
freqs = np.arange(N_pad) * sample_rate / N_pad
half = N_pad // 2

plt.figure(figsize=(7, 3))
plt.plot(freqs[:half], magnitude[:half])
plt.xlim(0, 1100); plt.xlabel('Frequency (Hz)'); plt.ylabel('Magnitude')
plt.title('Magnitude spectrum'); plt.grid(True, alpha=0.3); plt.show()

## 4. The winding view: what the transform actually does

Multiply the signal by $e^{-2\pi i f t}$ — wind it around the origin at 
frequency $f$ — and watch the center of mass. It bulges away from zero 
exactly when $f$ matches a frequency present in the signal.

In [ ]:
fs_d, dur_d = 1000, 2.0
td = np.arange(int(fs_d * dur_d)) / fs_d
g = np.cos(2*np.pi*2*td) + 0.7*np.cos(2*np.pi*3*td)  # 2 Hz + 3 Hz

def wound(signal, tt, f):
    return signal * np.exp(-2j * np.pi * f * tt)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), subplot_kw={'aspect': 'equal'})
for ax, f in zip(axes, [1.0, 2.0, 2.5, 3.0]):
    z = wound(g, td, f); com = z.mean()
    ax.plot(z.real, z.imag, lw=0.7)
    ax.plot(0, 0, 'k+'); ax.plot(com.real, com.imag, 'ro')
    ax.set_title(f'wind @ {f:g} Hz\n|COM|={abs(com):.2f}')
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

In [ ]:
sweep = np.linspace(0, 6, 600)
com_mag = np.array([abs(wound(g, td, f).mean()) for f in sweep])
plt.figure(figsize=(7, 3))
plt.plot(sweep, com_mag)
plt.xlabel('Winding frequency (Hz)'); plt.ylabel('|center of mass|')
plt.title('Sweep winding frequency = the Fourier transform')
plt.grid(True, alpha=0.3); plt.show()

## 5. Complexity: naive DFT vs FFT timing

In [ ]:
import time
sizes = [2**k for k in range(6, 13)]
naive_times, fft_times = [], []
for n in sizes:
    xx = np.random.default_rng(0).standard_normal(n)
    t0 = time.perf_counter(); naive_dft(xx); naive_times.append(time.perf_counter()-t0)
    t0 = time.perf_counter(); fft_recursive(xx); fft_times.append(time.perf_counter()-t0)

plt.figure(figsize=(7, 3.6))
plt.loglog(sizes, naive_times, 'o-', label='naive DFT  O(N^2)')
plt.loglog(sizes, fft_times, 'o-', label='recursive FFT  O(N log N)')
plt.xlabel('N'); plt.ylabel('Time (s)'); plt.legend()
plt.grid(True, which='both', alpha=0.3); plt.show()